# Experiment 1 — Data Quality Checks

**PDF reference: Phase 1, Experiment 1**

Checks before running the statistical pipeline:
- Power distribution per bike — are you riding at similar watts on each bike?
- Segment coverage matrix — which segments have efforts from multiple bikes?
- Effort count per segment per bike — filter out thin data.
- Stop criteria: fewer than ~10 paired segments → CIs will be wide.

Reads directly from the SQLite database so this notebook can be run independently of the Streamlit app.

In [ ]:
import sys
from pathlib import Path

# Add repo root to path
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.database import init_db, load_efforts, load_segments, load_bikes
from src.bike_delta import prepare_delta_dataset

sns.set_theme(style='whitegrid', palette='Set2')
print('Imports OK')

: 

In [ ]:
# ── Load data from SQLite ────────────────────────────────────────────────────
init_db()
raw_efforts = load_efforts()
segments = load_segments()
bikes_df = load_bikes()
bikes_dict = dict(zip(bikes_df['gear_id'], bikes_df['name'])) if not bikes_df.empty else {}

print(f'Efforts: {len(raw_efforts):,}')
print(f'Segments: {len(segments):,}')
print(f'Bikes: {bikes_dict}')

raw_efforts.head(3)

In [ ]:
# ── Prepare dataset ──────────────────────────────────────────────────────────
# Keep only efforts with power data
power_efforts = raw_efforts[raw_efforts['average_watts'].notna()].copy()
print(f'Efforts with power: {len(power_efforts):,}')

df = prepare_delta_dataset(power_efforts, segments, bikes_dict)
print(f'Analysis dataset rows: {len(df):,}')
print(f'Bikes present: {df["bike_name"].unique().tolist()}')
df.head(3)

## 1.1 — Power distribution per bike
Are you riding at similar watts on each bike? Large differences suggest the bikes are used in different contexts (e.g. one for training, one for racing), which would confound the comparison.

In [ ]:
# Summary statistics
print('=== Power distribution per bike (W) ===')
print(df.groupby('bike_name')['average_watts'].describe().round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box plot
sns.boxplot(data=df, x='bike_name', y='average_watts', ax=axes[0])
axes[0].set_title('Power distribution per bike')
axes[0].set_xlabel('Bike')
axes[0].set_ylabel('Avg power (W)')

# KDE overlay
for bike, grp in df.groupby('bike_name'):
    grp['average_watts'].dropna().plot.kde(ax=axes[1], label=bike)
axes[1].set_title('Power KDE per bike')
axes[1].set_xlabel('Avg power (W)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 1.2 — Segment coverage matrix
Which segments have efforts from multiple bikes? This is the pool from which paired comparisons can be drawn.

In [ ]:
# Segment × bike effort count pivot
pivot = (
    df.groupby(['segment_id', 'bike_name'])['effort_id']
    .count()
    .unstack(fill_value=0)
)

# Merge segment names
if 'name' in segments.columns:
    seg_names = segments[['segment_id', 'name']].drop_duplicates('segment_id')
    pivot = pivot.merge(seg_names, left_index=True, right_on='segment_id', how='left').set_index('name')
    pivot.index.name = 'Segment'

print(f'Segments with any efforts: {len(pivot)}')
pivot

In [ ]:
# Heatmap of effort counts
fig, ax = plt.subplots(figsize=(max(6, len(pivot.columns) * 2), max(4, len(pivot) * 0.5 + 2)))
sns.heatmap(
    pivot,
    annot=True, fmt='d', cmap='Blues',
    linewidths=0.5, linecolor='white',
    ax=ax
)
ax.set_title('Effort count per segment per bike')
plt.tight_layout()
plt.show()

## 1.3 — Paired segment count
How many segments have ≥ N efforts for ALL bikes? This is the effective analysis sample.

In [ ]:
from src.bike_delta import get_paired_segments

all_bikes = df['bike_name'].dropna().unique().tolist()
print(f'All bikes: {all_bikes}\n')

for min_n in [1, 2, 3, 5]:
    paired = get_paired_segments(df, all_bikes, min_efforts=min_n)
    print(f'min_efforts={min_n}: {len(paired)} paired segments')

# Stop criteria check
RECOMMENDED_MIN = 3
paired_recommended = get_paired_segments(df, all_bikes, min_efforts=RECOMMENDED_MIN)
print(f'\nAt recommended threshold (≥{RECOMMENDED_MIN} efforts): {len(paired_recommended)} paired segments')

if len(paired_recommended) < 10:
    print('⚠️  STOP CRITERIA: fewer than 10 paired segments. CIs will be wide.')
    print('    Consider: riding more segments on multiple bikes, or reducing min_efforts.')
else:
    print('✅  Sufficient paired segments for analysis.')

In [ ]:
# Show only the paired segments
paired_segs = get_paired_segments(df, all_bikes, min_efforts=RECOMMENDED_MIN)
paired_df = pivot.loc[pivot.index.isin(
    segments.loc[segments['segment_id'].isin(paired_segs), 'name'].values
    if 'name' in segments.columns else []
)]
print(f'Paired segments (≥{RECOMMENDED_MIN} efforts on all bikes):')
print(paired_df.to_string() if not paired_df.empty else '(no matching names found in pivot — check segment_id join)')